In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
import time
from src.features.materialize import apply_and_materialize
t = time.perf_counter()
store = apply_and_materialize("feature_store.yaml")
print(f"Applied and materialized in {time.perf_counter() - t:.1f}s")

In [ ]:
[fv.name for fv in store.list_feature_views()], [e.name for e in store.list_entities()]

In [ ]:
from src.config import USER_FEATURES
refs = [f"user_features:{f}" for f in USER_FEATURES]
store.get_online_features(features=refs, entity_rows=[{"user_id": 1}, {"user_id": 2}, {"user_id": 3}]).to_df()

In [ ]:
import numpy as np
lat = []
for uid in np.random.default_rng(0).integers(0, 20000, 300):
    s = time.perf_counter()
    store.get_online_features(features=refs, entity_rows=[{"user_id": int(uid)}])
    lat.append((time.perf_counter() - s) * 1000)
print(f"Online lookup p50 {np.percentile(lat, 50):.2f} ms p95 {np.percentile(lat, 95):.2f} ms p99 {np.percentile(lat, 99):.2f} ms")

In [ ]:
import pandas as pd
entity_df = pd.read_parquet("data/raw/transactions.parquet", columns=["user_id", "event_timestamp"]).head(1000)
historical = store.get_historical_features(entity_df=entity_df, features=store.get_feature_service("fraud_scoring_v1")).to_df()
historical.head()